In [7]:
import json
import re
from pathlib import Path
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

In [15]:
INPUT_PATH = Path("preprocessed_articles_100k.csv")   # <-- change to your Task 3 output file
OUTPUT_DIR = Path("validation_output")

# Logical field -> possible column names in your file (first match is used)
COLUMN_CANDIDATES = {
    "url":          ["news_url_clean", "url", "link"],
    "title":        ["title_clean", "title"],
    "content":      ["content_clean", "content", "description_clean", "description"],
    "published_at": ["published_at_utc", "published_at"],
    "source":       ["source", "domain"],
    "scraped_at":   ["scraped_at_utc", "scraped_at"],
}

TITLE_MIN, TITLE_MAX = 10, 65        # TITLE_MIN: confirm with your lecture note
CONTENT_MIN, CONTENT_MAX = 200, 2000

SEVERITY = {
    "TITLE_MISSING":    "ERROR",
    "TITLE_LENGTH":     "ERROR",
    "URL_MISSING":      "ERROR",
    "URL_INVALID":      "ERROR",
    "DUP_URL":          "ERROR",
    "URL_REDIRECT":     "WARNING",
    "URL_DEAD":         "WARNING",
    "CONTENT_MISSING":  "ERROR",
    "CONTENT_LENGTH":   "ERROR",
    "DUP_CROSS_SOURCE": "ERROR",     # same news on another website, published later
    "DUP_SAME_SOURCE":  "WARNING",   # same news repeated on the same website
    "TIME_INVALID":     "WARNING",
    "TITLE_LENGTH":     "WARNING",
}

CHECK_REDIRECTS = True   # set True to run the redirect check
REDIRECT_SAMPLE = 500     # None = check all valid URLs (very slow)
MAX_WORKERS = 32
REQUEST_TIMEOUT = 10

In [9]:
df = pd.read_csv(INPUT_PATH, low_memory=False).reset_index(drop=True)
df["row_id"] = df.index          # stable id used by the issue log

cols = {k: next((c for c in v if c in df.columns), None) for k, v in COLUMN_CANDIDATES.items()}
print("Column mapping:", cols)
print("Shape:", df.shape)
df.head()

Column mapping: {'url': 'url', 'title': 'title', 'content': 'description', 'published_at': 'published_at', 'source': 'source', 'scraped_at': 'scraped_at'}
Shape: (100000, 10)


,article_id,source,title,description,author,url,published_at,timezone,scraped_at,row_id
0,431f5566a31b29b4,BBC News,"Almost 1,000 arrested as clashes follow another day of mass protests across France","Hundreds of thousands took part in demonstrations in France, with protests also taking place in ...",NaN,https://www.bbc.co.uk/news/articles/c95ynd587ynzo?at_medium=RSS&at_campaign=rss,2026-10-25 04:16:52+00:00,UTC,2026-10-25 06:41:03.641673+00:00,0
1,ff00da6ff48d059a,BBC News,Israel to close West Bank crossings on election day as crackdown escalates,The Israeli Defense Forces announces restrictions on the movement of Palestinian workers in the ...,NaN,https://www.bbc.co.uk/news/articles/c6y93ylpl5p2o?at_medium=RSS&at_campaign=rss,2026-10-25 05:15:52+00:00,UTC,2026-10-25 06:41:07.354368+00:00,1
2,ee273e8d6d2effa2,BBC News,Voters in Brazil to choose between Lula and Bolsonaro's son,"Flávio Bolsonaro, the son of Brazil's imprisoned former president, is presenting a strong challe...",NaN,https://www.bbc.co.uk/news/articles/cxp8g21j4648o?at_medium=RSS&at_campaign=rss,2026-10-24 23:50:02+00:00,UTC,2026-10-25 06:41:11.347318+00:00,2
3,af154c8b7028fa6b,BBC News,Ukraine says Starlink connection 'now stable' after network limits,The slowed Starlink connections had affected the Ukraine's military and drone operations for a w...,NaN,https://www.bbc.co.uk/news/articles/cwkg5md14j8go?at_medium=RSS&at_campaign=rss,2026-10-24 12:12:40+00:00,UTC,2026-10-25 06:41:15.790533+00:00,3
4,35c67fdb10b3910b,BBC News,Gunmen kill at least 10 villagers and loot shops in northeast Nigeria,"Police are searching for other casualties, with officials warning the actual death toll could ri...",NaN,https://www.bbc.co.uk/news/articles/cq1kdlm1ze7do?at_medium=RSS&at_campaign=rss,2026-10-25 03:20:16+00:00,UTC,2026-10-25 06:41:20.034129+00:00,4


In [10]:
missing_required = [k for k in ("url", "title", "content") if cols[k] is None]
assert not missing_required, f"Missing required columns for: {missing_required}. Update COLUMN_CANDIDATES."
print("Schema OK: all required columns are present.")

Schema OK: all required columns are present.


In [11]:
issues = []

def flag(mask, rule, message):
    """Add one issue for every row where mask is True."""
    for i in df.index[mask]:
        issues.append({"row_id": int(i), "rule": rule,
                       "severity": SEVERITY[rule], "message": message})

def is_valid_http_url(value):
    if not isinstance(value, str) or not value.strip():
        return False
    try:
        p = urlparse(value.strip())
        return p.scheme.lower() in {"http", "https"} and bool(p.hostname) and "." in p.hostname
    except ValueError:
        return False

def blank(series):
    return series.isna() | (series.astype(str).str.strip() == "")

def norm_text(s):
    return re.sub(r"\W+", " ", str(s).casefold()).strip()

title, url, content = df[cols["title"]], df[cols["url"]], df[cols["content"]]

In [12]:
title_missing = blank(title)
flag(title_missing, "TITLE_MISSING", "Title is missing or blank.")

title_len = title.fillna("").astype(str).str.strip().str.len()
flag(~title_missing & ((title_len < TITLE_MIN) | (title_len > TITLE_MAX)),
     "TITLE_LENGTH", f"Title length must be {TITLE_MIN}-{TITLE_MAX} characters.")

print("Title issues:", sum(1 for i in issues if i["rule"].startswith("TITLE")))
title_len.describe()

Title issues: 65103


count    100000.00000
mean         75.85091
std          30.26117
min           4.00000
25%          58.00000
50%          75.00000
75%          93.00000
max         287.00000
Name: title, dtype: float64

In [13]:
url_missing = blank(url)
flag(url_missing, "URL_MISSING", "URL is missing or blank.")

url_valid = url.map(is_valid_http_url)
flag(~url_missing & ~url_valid, "URL_INVALID", "URL is not a valid http(s) URL.")
flag(url_valid & url.duplicated(keep="first"), "DUP_URL", "Same URL appears more than once.")

print("Valid URLs:", int(url_valid.sum()), "/", len(df))

Valid URLs: 100000 / 100000


In [16]:
df["http_status"] = None
df["final_url"] = None

def check_redirect(u):
    import requests
    headers = {"User-Agent": "Mozilla/5.0 (data-validation)"}
    try:
        r = requests.head(u, allow_redirects=True, timeout=REQUEST_TIMEOUT, headers=headers)
        if r.status_code in (403, 405, 501):
            r = requests.get(u, allow_redirects=True, timeout=REQUEST_TIMEOUT, headers=headers, stream=True)
        return r.status_code, r.url, None
    except Exception as exc:
        return None, None, type(exc).__name__

if CHECK_REDIRECTS:
    targets = df[url_valid]
    if REDIRECT_SAMPLE:
        targets = targets.sample(min(REDIRECT_SAMPLE, len(targets)), random_state=42)
    urls = targets[cols["url"]].str.strip().tolist()
    print(f"Checking {len(urls)} URLs ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        results = list(ex.map(check_redirect, urls))
    for i, u, (status, final, err) in zip(targets.index, urls, results):
        df.at[i, "http_status"], df.at[i, "final_url"] = status, final
        if err or (status is not None and status >= 400):
            issues.append({"row_id": int(i), "rule": "URL_DEAD", "severity": SEVERITY["URL_DEAD"],
                           "message": f"URL not reachable (status={status}, error={err})."})
        elif final and final.rstrip("/") != u.rstrip("/"):
            issues.append({"row_id": int(i), "rule": "URL_REDIRECT", "severity": SEVERITY["URL_REDIRECT"],
                           "message": f"URL redirects to {final}"})
    print("Redirect check finished.")
else:
    print("Redirect check skipped (CHECK_REDIRECTS = False).")

Checking 500 URLs ...
Redirect check finished.


In [17]:
check_df = pd.DataFrame(issues)
redir = check_df[check_df["rule"].isin(["URL_REDIRECT", "URL_DEAD"])]

print("URLs checked      :", df["http_status"].notna().sum() + (df["http_status"].isna() & df["final_url"].isna() & df.row_id.isin(redir.row_id)).sum())
print("Redirected (warn) :", (redir["rule"] == "URL_REDIRECT").sum())
print("Dead / error (warn):", (redir["rule"] == "URL_DEAD").sum())
print()
print("HTTP status counts:")
print(df["http_status"].value_counts(dropna=False).head(10).to_string())
print()
redir.head(10)

URLs checked      : 500
Redirected (warn) : 289
Dead / error (warn): 59

HTTP status counts:
http_status
None    99509
200       441
403        27
406        21
502         2



,row_id,rule,severity,message
65103,75721,URL_REDIRECT,WARNING,URL redirects to https://news.google.com/rss/articles/CBMiX0FVX3lxTE1oT1h4VUhoenZmdmpDVl9fWG1WT0...
65104,80184,URL_REDIRECT,WARNING,URL redirects to https://news.google.com/rss/articles/CBMiVEFVX3lxTFBTSFNEOENDUnREOHJ1OG1GS3ZGST...
65105,76699,URL_REDIRECT,WARNING,URL redirects to https://news.google.com/rss/articles/CBMiaEFVX3lxTE54R3JtLUhlQXZnQk5Bb0s2OTF2Vk...
65106,92991,URL_REDIRECT,WARNING,URL redirects to https://news.google.com/rss/articles/CBMingFBVV95cUxPNnNqTGVXYzZMZ1lFcHp2UGN0Nk...
65107,76434,URL_REDIRECT,WARNING,URL redirects to https://news.google.com/rss/articles/CBMitAFBVV95cUxQV0FMZEFnd0pfNG1scVBId3YyQX...
65108,84004,URL_REDIRECT,WARNING,URL redirects to https://news.google.com/rss/articles/CBMirgFBVV95cUxPc1FPd2htS1NVeUgzb0FsQnRzaV...
65109,80917,URL_REDIRECT,WARNING,URL redirects to https://news.google.com/rss/articles/CBMidEFVX3lxTE9KX2c3bDJEczROVVJlb3ZMM2xXak...
65110,60767,URL_REDIRECT,WARNING,URL redirects to https://news.google.com/rss/articles/CBMiWEFVX3lxTE9STGhQQk5LU01MNTBkcHdtNWNrME...
65111,50074,URL_REDIRECT,WARNING,URL redirects to https://news.google.com/rss/articles/CBMieEFVX3lxTE96MG1sSUhwU2xWMVppMV95d3piaE...
65112,42141,URL_DEAD,WARNING,"URL not reachable (status=403, error=None)."


In [18]:
# Content validation
content = df[cols["content"]]

content_missing = blank(content)

flag(
    content_missing,
    "CONTENT_MISSING",
    "Description/content is missing or blank."
)

content_len = content.fillna("").astype(str).str.strip().str.len()

# Descriptions are usually shorter than full article text.
CONTENT_MIN = 200
CONTENT_MAX = 2000

flag(
    ~content_missing &
    ((content_len < CONTENT_MIN) | (content_len > CONTENT_MAX)),
    "CONTENT_LENGTH",
    f"Description length must be {CONTENT_MIN}-{CONTENT_MAX} characters."
)

print("Missing descriptions:", int(content_missing.sum()))
print("Short/long descriptions:", int(
    ((~content_missing) &
     ((content_len < CONTENT_MIN) | (content_len > CONTENT_MAX))).sum()
))
print(content_len.describe())

Missing descriptions: 123
Short/long descriptions: 93315
count    100000.000000
mean        113.858040
std         137.643545
min           0.000000
25%          66.000000
50%          87.000000
75%         117.000000
max        7950.000000
Name: description, dtype: float64


In [19]:
CONTENT_MIN = 200
CONTENT_MAX = 2000

content_len = (
    df[cols["content"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.len()
)

content_missing = blank(df[cols["content"]])

short_desc = ~content_missing & (content_len < CONTENT_MIN)
long_desc = content_len > CONTENT_MAX

print("Missing descriptions:", int(content_missing.sum()))
print("Short descriptions (<200):", int(short_desc.sum()))
print("Long descriptions (>2000):", int(long_desc.sum()))
print(
    "Total short/long:",
    int((short_desc | long_desc).sum())
)

Missing descriptions: 123
Short descriptions (<200): 93281
Long descriptions (>2000): 34
Total short/long: 93315


In [20]:
# Normalize titles for duplicate detection
df["normalized_title"] = (
    df[cols["title"]]
    .fillna("")
    .astype(str)
    .map(norm_text)
)

source_col = cols["source"]

valid_title = df["normalized_title"].ne("")

# Same normalized title from the same source
same_source_dup = (
    valid_title &
    df.duplicated(
        subset=["normalized_title", source_col],
        keep="first"
    )
)

flag(
    same_source_dup,
    "DUP_SAME_SOURCE",
    "Repeated normalized title within the same source."
)

# Same normalized title from a different source
cross_source_dup = (
    valid_title &
    df.duplicated(
        subset=["normalized_title"],
        keep="first"
    ) &
    ~same_source_dup
)

flag(
    cross_source_dup,
    "DUP_CROSS_SOURCE",
    "Normalized title appears in multiple source records; review whether it is the same news story."
)

print("Same-source duplicate candidates:", int(same_source_dup.sum()))
print("Cross-source duplicate candidates:", int(cross_source_dup.sum()))

Same-source duplicate candidates: 552
Cross-source duplicate candidates: 291


In [21]:
# Check publication and scraping timestamps
for field in ["published_at", "scraped_at"]:
    col = cols[field]

    if col is None:
        print(f"{field}: column not found; skipped.")
        continue

    parsed_time = pd.to_datetime(
        df[col],
        errors="coerce",
        utc=True
    )

    invalid = df[col].notna() & parsed_time.isna()
    missing_time = blank(df[col])

    # Scraping timestamps should not be in the future.
    future_time = (
        parsed_time.notna() &
        (parsed_time > pd.Timestamp.now(tz="UTC"))
        if field == "scraped_at"
        else pd.Series(False, index=df.index)
    )

    print(f"\n{field}")
    print("Missing:", int(missing_time.sum()))
    print("Invalid:", int(invalid.sum()))
    print("Future timestamps:", int(future_time.sum()))

    if field == "published_at":
        flag(
            invalid,
            "TIME_INVALID",
            "Publication timestamp could not be parsed."
        )
    else:
        flag(
            invalid | future_time,
            "TIME_INVALID",
            "Scraping timestamp is invalid or in the future."
        )


published_at
Missing: 0
Invalid: 8225
Future timestamps: 0

scraped_at
Missing: 0
Invalid: 849
Future timestamps: 0


In [22]:
for field in ["published_at", "scraped_at"]:
    col = cols[field]

    parsed = pd.to_datetime(
        df[col],
        errors="coerce",
        utc=True
    )

    invalid_mask = df[col].notna() & parsed.isna()

    print(f"\nInvalid examples: {field}")
    print(
        df.loc[invalid_mask, [col]]
        .head(10)
        .to_string(index=False)
    )


Invalid examples: published_at
                    published_at
2026-10-24 16:32:03.638000+00:00
2026-10-24 21:39:13.133000+00:00
2026-10-24 23:50:02.473000+00:00
2026-10-24 23:04:17.565000+00:00
2026-10-25 04:00:15.948000+00:00
2026-10-25 04:51:10.385000+00:00
2026-10-24 05:01:03.315000+00:00
2026-08-18 18:42:17.416000+00:00
2026-10-25 00:56:44.168000+00:00
2026-10-24 07:29:25.853000+00:00

Invalid examples: scraped_at
               scraped_at
2026-10-21 15:26:27+00:00
2026-10-21 15:26:28+00:00
2026-10-21 15:26:29+00:00
2026-10-21 15:26:30+00:00
2026-10-21 15:26:31+00:00
2026-10-21 15:26:32+00:00
2026-10-21 15:26:33+00:00
2026-10-21 15:26:33+00:00
2026-10-21 15:26:34+00:00
2026-10-21 15:26:35+00:00


In [23]:
for field in ["published_at", "scraped_at"]:
    col = cols[field]

    parsed = pd.to_datetime(
        df[col],
        format="mixed",
        errors="coerce",
        utc=True
    )

    missing = blank(df[col])
    invalid = ~missing & parsed.isna()
    future = parsed.notna() & (
        parsed > pd.Timestamp.now(tz="UTC")
    )

    print(f"\n{field}")
    print("Missing:", int(missing.sum()))
    print("Invalid:", int(invalid.sum()))
    print("Future:", int(future.sum()))

    print("\nRemaining invalid examples:")
    print(df.loc[invalid, col].head(10).to_string(index=False))


published_at
Missing: 0
Invalid: 0
Future: 0

Remaining invalid examples:
Series([], )

scraped_at
Missing: 0
Invalid: 0
Future: 0

Remaining invalid examples:
Series([], )


In [24]:
# Remove outdated timestamp findings from the earlier parsing attempt
issues = [
    issue for issue in issues
    if issue["rule"] != "TIME_INVALID"
]

# Create the output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Build the updated issue log
issues_df = pd.DataFrame(
    issues,
    columns=["row_id", "rule", "severity", "message"]
)

# Save detailed validation issues
issues_df.to_csv(
    OUTPUT_DIR / "validation_issues.csv",
    index=False
)

# Generate summary by rule and severity
if not issues_df.empty:
    summary_df = (
        issues_df
        .groupby(["rule", "severity"])
        .size()
        .reset_index(name="issue_count")
        .sort_values("issue_count", ascending=False)
    )
else:
    summary_df = pd.DataFrame(
        columns=["rule", "severity", "issue_count"]
    )

summary_df.to_csv(
    OUTPUT_DIR / "validation_summary.csv",
    index=False
)

print("Validation issue records:", len(issues_df))
print("\nFinal issue summary:")
print(summary_df.to_string(index=False))

print("\nFiles saved:")
print(OUTPUT_DIR / "validation_issues.csv")
print(OUTPUT_DIR / "validation_summary.csv")

Validation issue records: 159732

Final issue summary:
            rule severity  issue_count
  CONTENT_LENGTH    ERROR        93315
    TITLE_LENGTH  WARNING        65103
 DUP_SAME_SOURCE  WARNING          552
DUP_CROSS_SOURCE    ERROR          291
    URL_REDIRECT  WARNING          289
 CONTENT_MISSING    ERROR          123
        URL_DEAD  WARNING           59

Files saved:
validation_output\validation_issues.csv
validation_output\validation_summary.csv


In [25]:
# Align severity with the lecture's review-based approach
for issue in issues:
    if issue["rule"] in ["CONTENT_LENGTH", "DUP_CROSS_SOURCE"]:
        issue["severity"] = "WARNING"

# Keep the existing rules:
# Missing content -> ERROR
# Title length -> WARNING
# Same-source duplicates -> WARNING
# URL redirects/dead responses -> WARNING

issues_df = pd.DataFrame(
    issues,
    columns=["row_id", "rule", "severity", "message"]
)

summary_df = (
    issues_df
    .groupby(["rule", "severity"])
    .size()
    .reset_index(name="issue_count")
    .sort_values("issue_count", ascending=False)
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

issues_df.to_csv(
    OUTPUT_DIR / "validation_issues.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR / "validation_summary.csv",
    index=False
)

print("Updated validation summary:")
print(summary_df.to_string(index=False))

print("\nTotal issue records:", len(issues_df))

Updated validation summary:
            rule severity  issue_count
  CONTENT_LENGTH  WARNING        93315
    TITLE_LENGTH  WARNING        65103
 DUP_SAME_SOURCE  WARNING          552
DUP_CROSS_SOURCE  WARNING          291
    URL_REDIRECT  WARNING          289
 CONTENT_MISSING    ERROR          123
        URL_DEAD  WARNING           59

Total issue records: 159732


In [26]:
# Find articles with missing descriptions
content_col = cols["content"]
source_col = cols["source"]
title_col = cols["title"]
url_col = cols["url"]

missing_mask = blank(df[content_col])

review_columns = [
    "row_id",
    source_col,
    title_col,
    url_col,
    content_col
]

missing_review = df.loc[
    missing_mask, review_columns
].copy()

# Save all missing-description records for review
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing_review.to_csv(
    OUTPUT_DIR / "missing_description_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Missing description records:", len(missing_review))

print("\nMissing descriptions by source:")
print(
    missing_review[source_col]
    .fillna("Unknown")
    .value_counts()
    .to_string()
)

print("\nFirst 10 records:")
print(
    missing_review[
        ["row_id", source_col, title_col, url_col]
    ].head(10).to_string(index=False)
)

Missing description records: 123

Missing descriptions by source:
source
The Independent    90
Politico            9
NYT World           6
NPR                 5
CBC News            5
NYT                 4
New York Times      1
Wired               1
CNET                1
France 24           1

First 10 records:
 row_id          source                                                                         title                                                                                                                url
    100  New York Times                                                            Here’s the latest.                               https://www.nytimes.com/live/2026/10/24/world/france-protests-riots/heres-the-latest
    878 The Independent    Airbnb agrees guests shouldn’t have to do the dishes or strip beds anymore         https://www.independent.co.uk/bulletin/lifestyle/airbnb-guest-rules-changing-cleaning-chores-b3072812.html
    935 The Independent           Kri

In [27]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor

# Work only on records with missing descriptions
content_col = cols["content"]
missing_mask = blank(df[content_col])

missing_df = df.loc[missing_mask].copy()

def recover_description(row):
    url = str(row[cols["url"]]).strip()

    result = {
        "row_id": row["row_id"],
        "recovered_description": "",
        "recovery_status": ""
    }

    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=12
        )

        if response.status_code != 200:
            result["recovery_status"] = (
                f"HTTP_{response.status_code}"
            )
            return result

        soup = BeautifulSoup(response.text, "html.parser")

        # Try common metadata fields
        description = ""

        for selector in [
            {"property": "og:description"},
            {"name": "description"},
            {"name": "twitter:description"}
        ]:
            tag = soup.find("meta", attrs=selector)

            if tag and tag.get("content", "").strip():
                description = tag["content"].strip()
                break

        if description:
            result["recovered_description"] = description
            result["recovery_status"] = "RECOVERED"
        else:
            result["recovery_status"] = "NO_DESCRIPTION_FOUND"

    except requests.RequestException as exc:
        result["recovery_status"] = type(exc).__name__

    return result


with ThreadPoolExecutor(max_workers=5) as executor:
    recovered_results = list(
        executor.map(
            recover_description,
            [row for _, row in missing_df.iterrows()]
        )
    )

recovery_df = pd.DataFrame(recovered_results)

recovery_df.to_csv(
    OUTPUT_DIR / "description_recovery_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total checked:", len(recovery_df))
print("\nRecovery results:")
print(recovery_df["recovery_status"].value_counts().to_string())

Total checked: 123

Recovery results:
recovery_status
NO_DESCRIPTION_FOUND    108
ConnectionError          10
RECOVERED                 5


In [28]:
import pandas as pd
from pathlib import Path

recovery_file = Path(
    "validation_output/description_recovery_results.csv"
)

recovery_df = pd.read_csv(recovery_file)

recovered = recovery_df[
    recovery_df["recovery_status"] == "RECOVERED"
]

print("Successfully recovered:", len(recovered))
print(
    recovered[
        ["row_id", "recovered_description", "recovery_status"]
    ].to_string(index=False)
)

Successfully recovered: 5
 row_id                                                                                                                                                                                                                                                                                                recovered_description recovery_status
  42294                                                                                           B.C. premier-designate Lorne Doerkson spent his victory speech Saturday night introducing himself to British Columbians, and celebrating the "trailblazers" who came to the province searching for gold and a better life.       RECOVERED
  42295                                                                  Prime Minister Mark Carney sees Turkey's rapidly expanding defence industry as a model for breaking Canada's dependence on American military suppliers. But a history of broken trust and political repression casts a shadow over the part

In [29]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("validation_output")

# Load original preprocessed dataset
df = pd.read_csv("preprocessed_articles_100k.csv")

# Load recovery results
recovery_df = pd.read_csv(
    OUTPUT_DIR / "description_recovery_results.csv"
)

# Keep successfully recovered descriptions
recovered = recovery_df[
    recovery_df["recovery_status"] == "RECOVERED"
].copy()

# Match recovered records with the original dataset
df["row_id"] = df.index

verified = recovered.merge(
    df[["row_id", "source", "title", "url", "description"]],
    on="row_id",
    how="left"
)

# Review original and recovered descriptions together
columns_to_review = [
    "row_id",
    "source",
    "title",
    "url",
    "description",
    "recovered_description"
]

print(verified[columns_to_review].to_string(index=False))

# Save a separate review file
verified[columns_to_review].to_csv(
    OUTPUT_DIR / "recovered_descriptions_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nReview file saved successfully.")

C:\Users\DELL\AppData\Local\Temp\ipykernel_4928\3777893562.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("preprocessed_articles_100k.csv")


 row_id   source                                                                                                 title                                                                                                            url description                                                                                                                                                                                                                                                                                                recovered_description
  42294 CBC News                                                             Who is B.C.'s new premier Lorne Doerkson?             https://www.cbc.ca/news/canada/british-columbia/who-is-bc-premier-lorne-doerkson-9.7395130?cmp=rss         NaN                                                                                           B.C. premier-designate Lorne Doerkson spent his victory speech Saturday night introducing himself to British Columbia

In [30]:
import pandas as pd
from pathlib import Path

output_dir = Path("validation_output")

review_path = output_dir / "recovered_descriptions_review.csv"

review_df = pd.read_csv(review_path, low_memory=False)

# Basic checks
review_df["recovered_description"] = (
    review_df["recovered_description"]
    .fillna("")
    .astype(str)
    .str.strip()
)

review_df["title"] = (
    review_df["title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

review_df["url"] = (
    review_df["url"]
    .fillna("")
    .astype(str)
    .str.strip()
)

review_df["verification_status"] = review_df.apply(
    lambda row: (
        "READY_FOR_MANUAL_REVIEW"
        if row["title"]
        and row["url"].startswith("https://")
        and len(row["recovered_description"]) > 0
        else "REVIEW_REQUIRED"
    ),
    axis=1
)

review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print(review_df[
    ["row_id", "source", "verification_status"]
].to_string(index=False))

print("\nVerification file updated:", review_path)

 row_id   source     verification_status
  42294 CBC News READY_FOR_MANUAL_REVIEW
  42295 CBC News READY_FOR_MANUAL_REVIEW
  42296 CBC News READY_FOR_MANUAL_REVIEW
  42302 CBC News READY_FOR_MANUAL_REVIEW
  42312 CBC News READY_FOR_MANUAL_REVIEW

Verification file updated: validation_output\recovered_descriptions_review.csv


In [31]:
print(issues.shape)
print(issues.columns.tolist())
print(issues.head())

AttributeError: 'list' object has no attribute 'shape'

In [32]:
print("Data type:", type(issues))
print("Total issue records:", len(issues))

print("\nFirst 2 records:")
print(issues[:2])

Data type: <class 'list'>
Total issue records: 159732

First 2 records:
[{'row_id': 0, 'rule': 'TITLE_LENGTH', 'severity': 'WARNING', 'message': 'Title length must be 10-65 characters.'}, {'row_id': 1, 'rule': 'TITLE_LENGTH', 'severity': 'WARNING', 'message': 'Title length must be 10-65 characters.'}]


In [33]:
import pandas as pd

issues_df = pd.DataFrame(issues)

print("Shape:", issues_df.shape)
print("Columns:", issues_df.columns.tolist())
print(issues_df.head())

Shape: (159732, 4)
Columns: ['row_id', 'rule', 'severity', 'message']
   row_id          rule severity                                 message
0       0  TITLE_LENGTH  WARNING  Title length must be 10-65 characters.
1       1  TITLE_LENGTH  WARNING  Title length must be 10-65 characters.
2       3  TITLE_LENGTH  WARNING  Title length must be 10-65 characters.
3       4  TITLE_LENGTH  WARNING  Title length must be 10-65 characters.
4       5  TITLE_LENGTH  WARNING  Title length must be 10-65 characters.


In [34]:
# Count issues by rule and severity
issue_summary = (
    issues_df
    .groupby(["rule", "severity"])
    .size()
    .reset_index(name="issue_count")
    .sort_values("issue_count", ascending=False)
)

print(issue_summary.to_string(index=False))

print("\nTotal issue records:", len(issues_df))

print("\nSeverity counts:")
print(issues_df["severity"].value_counts(dropna=False))

            rule severity  issue_count
  CONTENT_LENGTH  WARNING        93315
    TITLE_LENGTH  WARNING        65103
 DUP_SAME_SOURCE  WARNING          552
DUP_CROSS_SOURCE  WARNING          291
    URL_REDIRECT  WARNING          289
 CONTENT_MISSING    ERROR          123
        URL_DEAD  WARNING           59

Total issue records: 159732

Severity counts:
severity
WARNING    159609
ERROR         123
Name: count, dtype: int64


In [35]:
print("Total dataset rows:", len(df))
print("Unique row IDs:", df["row_id"].nunique())
print("Missing row IDs:", df["row_id"].isna().sum())

Total dataset rows: 100000
Unique row IDs: 100000
Missing row IDs: 0


In [36]:
print("Total dataset rows:", len(df))
print("Unique row IDs:", df["row_id"].nunique())
print("Missing row IDs:", df["row_id"].isna().sum())
print("Duplicate row IDs:", df["row_id"].duplicated().sum())

Total dataset rows: 100000
Unique row IDs: 100000
Missing row IDs: 0
Duplicate row IDs: 0


In [37]:

import pandas as pd
from pathlib import Path

# Output folder
OUTPUT_DIR = Path("validation_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Work on copies so original data remains unchanged
data = df.copy()
issue_data = issues_df.copy()

# Ensure row IDs are integers
data["row_id"] = pd.to_numeric(data["row_id"], errors="raise").astype(int)
issue_data["row_id"] = pd.to_numeric(
    issue_data["row_id"], errors="raise"
).astype(int)

# Confirm issue IDs belong to the dataset
unknown_ids = set(issue_data["row_id"]) - set(data["row_id"])

if unknown_ids:
    raise ValueError(
        f"{len(unknown_ids)} issue records have unknown row IDs."
    )

# Count issues per article
issue_counts = (
    issue_data.groupby("row_id")
    .agg(
        issue_count=("rule", "size"),
        error_count=("severity", lambda s: s.eq("ERROR").sum()),
        warning_count=("severity", lambda s: s.eq("WARNING").sum())
    )
    .reset_index()
)

# Add counts to every article
validation = data.merge(
    issue_counts,
    on="row_id",
    how="left",
    validate="one_to_one"
)

for col in ["issue_count", "error_count", "warning_count"]:
    validation[col] = validation[col].fillna(0).astype(int)

# Quarantine rule: required content is missing
# Other ERROR rules, if added later, will also trigger quarantine.
validation["disposition"] = "ACCEPT"

validation.loc[
    validation["warning_count"] > 0,
    "disposition"
] = "WARNING"

validation.loc[
    validation["error_count"] > 0,
    "disposition"
] = "QUARANTINE"

# Export complete dataset with disposition
validation.to_csv(
    OUTPUT_DIR / "news_validation_all.csv",
    index=False,
    encoding="utf-8-sig"
)

# Export separate files
validation[
    validation["disposition"] == "ACCEPT"
].to_csv(
    OUTPUT_DIR / "news_accepted.csv",
    index=False,
    encoding="utf-8-sig"
)

validation[
    validation["disposition"] == "WARNING"
].to_csv(
    OUTPUT_DIR / "news_warning_review.csv",
    index=False,
    encoding="utf-8-sig"
)

validation[
    validation["disposition"] == "QUARANTINE"
].to_csv(
    OUTPUT_DIR / "news_quarantine.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save issue details separately
issue_data.to_csv(
    OUTPUT_DIR / "validation_issues.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save final summary
disposition_summary = (
    validation["disposition"]
    .value_counts()
    .reindex(["ACCEPT", "WARNING", "QUARANTINE"], fill_value=0)
    .rename_axis("disposition")
    .reset_index(name="article_count")
)

disposition_summary.to_csv(
    OUTPUT_DIR / "validation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("DISPOSITION SUMMARY")
print(disposition_summary.to_string(index=False))

print("\nTotal articles:", len(validation))
print("Unique row IDs:", validation["row_id"].nunique())
print("Total issue records:", len(issue_data))
print("\nFiles saved in:", OUTPUT_DIR.resolve())


DISPOSITION SUMMARY
disposition  article_count
     ACCEPT           3161
    WARNING          96716
 QUARANTINE            123

Total articles: 100000
Unique row IDs: 100000
Total issue records: 159732

Files saved in: C:\Users\DELL\validation_output


In [38]:
# Confirm all articles are accounted for
assert len(validation) == 100_000
assert validation["row_id"].is_unique
assert validation["disposition"].notna().all()

# Confirm no article is assigned multiple dispositions
assert validation["row_id"].nunique() == 100_000

# Confirm quarantine articles have errors
quarantine = validation[
    validation["disposition"] == "QUARANTINE"
]

assert (quarantine["error_count"] > 0).all()

print("All final validation checks passed.")
print("Total articles:", len(validation))
print("Quarantine articles:", len(quarantine))

All final validation checks passed.
Total articles: 100000
Quarantine articles: 123


In [1]:

import pandas as pd
from pathlib import Path

base = Path(r"C:\Users\DELL\validation_output")

review = pd.read_csv(
    base / "recovered_descriptions_review.csv",
    dtype=str,
    keep_default_na=False
)

for i, row in review.iterrows():
    print(f"\n{i + 1}. {row['title']}")
    print("URL:", row["url"])
    print("Recovered description:", row["recovered_description"])



1. Who is B.C.'s new premier Lorne Doerkson?
URL: https://www.cbc.ca/news/canada/british-columbia/who-is-bc-premier-lorne-doerkson-9.7395130?cmp=rss
Recovered description: B.C. premier-designate Lorne Doerkson spent his victory speech Saturday night introducing himself to British Columbians, and celebrating the "trailblazers" who came to the province searching for gold and a better life.

2. Carney's geopolitical ambitions tested in defence industry co-operation with Turkey
URL: https://www.cbc.ca/news/politics/turkey-canada-defence-military-drones-technology-9.7395400?cmp=rss
Recovered description: Prime Minister Mark Carney sees Turkey's rapidly expanding defence industry as a model for breaking Canada's dependence on American military suppliers. But a history of broken trust and political repression casts a shadow over the partnership.

3. Danielle Smith scores slim majority on immigration questions, Albertans reject constitutional changes
URL: https://www.cbc.ca/news/canada/calgar

In [2]:

import pandas as pd
from pathlib import Path

# Folder paths
BASE_DIR = Path(r"C:\Users\DELL\validation_output")

ORIGINAL_FILE = BASE_DIR / "preprocessed_articles_100k.csv"
REVIEW_FILE = BASE_DIR / "recovered_descriptions_review.csv"
MATCH_FILE = BASE_DIR / "recovered_descriptions_match_check.csv"

OUTPUT_FILE = BASE_DIR / "preprocessed_articles_100k_recovered.csv"
LOG_FILE = BASE_DIR / "recovered_descriptions_update_log.csv"

# Load files as strings to preserve article IDs
df = pd.read_csv(
    ORIGINAL_FILE, dtype=str, keep_default_na=False, low_memory=False
)
review = pd.read_csv(
    REVIEW_FILE, dtype=str, keep_default_na=False, low_memory=False
)
matches = pd.read_csv(
    MATCH_FILE, dtype=str, keep_default_na=False, low_memory=False
)

# Keep only records whose source + title + URL matched exactly
matches = matches[
    matches["EXACT_MATCH"].str.strip().str.lower().eq("true")
].copy()

# Combine the recovery details with the verified matching information
recovery = review.merge(
    matches[["row_id", "matched_article_id", "EXACT_MATCH"]],
    on="row_id",
    how="inner",
    validate="one_to_one"
)

# Ensure article IDs are unique in the original dataset
if df["article_id"].duplicated().any():
    raise ValueError("Duplicate article IDs found. No update performed.")

updated_records = []

for _, item in recovery.iterrows():
    article_id = item["matched_article_id"]
    recovered_text = item["recovered_description"].strip()

    if not recovered_text:
        continue

    # Locate exactly one matching article ID
    row_mask = df["article_id"].eq(article_id)

    if row_mask.sum() != 1:
        print(f"SKIPPED: ID did not match exactly one row: {article_id}")
        continue

    original_text = df.loc[row_mask, "description"].iloc[0]

    # Fill only missing descriptions; do not replace existing text
    if original_text.strip() == "":
        df.loc[row_mask, "description"] = recovered_text

        updated_records.append({
            "article_id": article_id,
            "title": item["title"],
            "old_description": original_text,
            "new_description": recovered_text,
            "update_status": "DESCRIPTION_ADDED"
        })
    else:
        print(f"SKIPPED: Description already exists for {article_id}")

# Save a separate updated dataset and audit log
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
pd.DataFrame(updated_records).to_csv(
    LOG_FILE, index=False, encoding="utf-8-sig"
)

print("\nUPDATE COMPLETE")
print("Original rows:", 100000)
print("Updated descriptions:", len(updated_records))
print("New dataset:", OUTPUT_FILE)
print("Update log:", LOG_FILE)
print("New blank descriptions:", df["description"].str.strip().eq("").sum())



UPDATE COMPLETE
Original rows: 100000
Updated descriptions: 5
New dataset: C:\Users\DELL\validation_output\preprocessed_articles_100k_recovered.csv
Update log: C:\Users\DELL\validation_output\recovered_descriptions_update_log.csv
New blank descriptions: 118
